# API Data Freshness Check

Most recent available timestamp for each training data source, with lag in hours.

In [23]:
import io, zipfile
import requests
import pandas as pd

NOW = pd.Timestamp.now()
print(f"Now: {NOW.strftime('%Y-%m-%d %H:%M')}")

Now: 2026-04-04 09:01


In [24]:
# Open-Meteo — ERA5 archive, typically ~5-day lag; hourly resolution
r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 40.7128, "longitude": -74.0060,
    "start_date": (NOW - pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
    "end_date":   NOW.strftime("%Y-%m-%d"),
    "hourly": "temperature_2m",
    "timezone": "America/New_York",
})
r.raise_for_status()
h = r.json()["hourly"]
temps = pd.Series(h["temperature_2m"], index=pd.to_datetime(h["time"]))
latest_openmeteo = temps.last_valid_index()
lag_h = (NOW - latest_openmeteo).total_seconds() / 3600
print(f"Open-Meteo:  {latest_openmeteo}  ({lag_h:.0f}h lag)")

Open-Meteo:  2026-04-04 23:00:00  (-14h lag)


In [25]:
# NYISO — 5-min interval data; walk back up to 3 months until a zip is available
latest_nyiso = None
for p in pd.period_range(end=NOW, periods=3, freq="M")[::-1]:
    r = requests.get(
        f"https://mis.nyiso.com/public/csv/pal/{p.year}{p.month:02d}01pal_csv.zip",
        timeout=15,
    )
    if r.ok:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            raw = pd.concat([pd.read_csv(z.open(n)) for n in z.namelist()])
        latest_nyiso = pd.to_datetime(raw["Time Stamp"]).max()
        lag_h = (NOW - latest_nyiso).total_seconds() / 3600
        print(f"NYISO:       {latest_nyiso}  ({lag_h:.0f}h lag)")
        break

NYISO:       2026-04-04 08:45:00  (0h lag)


In [26]:
# MTA daily ridership (city-wide total) — sayj-mze2
r = requests.get("https://data.ny.gov/resource/sayj-mze2.json",
                 params={"$order": "date DESC", "$limit": "1"}, timeout=30)
r.raise_for_status()
latest_mta_daily = pd.to_datetime(r.json()[0]["date"])
lag_h = (NOW - latest_mta_daily).total_seconds() / 3600
print(f"MTA:   {latest_mta_daily}  ({lag_h:.0f}h lag)")
# Note: MTA hourly-by-borough dataset (f462-ka72 / 5wq4-mkjj) was evaluated
# but has a ~180h publication lag — too stale for production use (8-day feature).

# NYC 311 — sub-hour timestamp precision
r = requests.get("https://data.cityofnewyork.us/resource/erm2-nwe9.json",
                 params={"$order": "created_date DESC", "$limit": "1"}, timeout=30)
r.raise_for_status()
latest_311 = pd.to_datetime(r.json()[0]["created_date"])
lag_h = (NOW - latest_311).total_seconds() / 3600
print(f"311:   {latest_311}  ({lag_h:.0f}h lag)")

MTA:   2026-04-02 00:00:00  (57h lag)
311:   2026-04-03 03:04:27  (30h lag)


In [27]:
# NYC Motor Vehicle Crashes — Socrata h9gi-nx95, server-side daily aggregation
r = requests.get("https://data.cityofnewyork.us/resource/h9gi-nx95.json", params={
    "$order": "crash_date DESC",
    "$limit": "1",
}, timeout=30)
r.raise_for_status()
latest_crashes = pd.to_datetime(r.json()[0]["crash_date"])
lag_h = (NOW - latest_crashes).total_seconds() / 3600
print(f"Crashes:     {latest_crashes}  ({lag_h:.0f}h lag)")

# Rate limit check — time a realistic aggregated query (one month of daily counts)
import time
t0 = time.perf_counter()
r2 = requests.get("https://data.cityofnewyork.us/resource/h9gi-nx95.json", params={
    "$select": "date_trunc_ymd(crash_date) as date, count(*) as crashes, "
               "sum(case(contributing_factor_vehicle_1='Pavement Slippery',1,true,0)) as slippery",
    "$group":  "date_trunc_ymd(crash_date)",
    "$where":  "crash_date >= '2024-12-01T00:00:00' AND crash_date <= '2024-12-31T23:59:59'",
    "$order":  "date ASC",
    "$limit":  "50",
}, timeout=30)
elapsed = time.perf_counter() - t0
r2.raise_for_status()
sample = pd.DataFrame(r2.json())
sample["date"] = pd.to_datetime(sample["date"]).dt.normalize()
print(f"  aggregated query: {len(sample)} days in {elapsed:.2f}s")
print(f"  sample: {sample.head(3).to_dict('records')}")

Crashes:     2026-03-31 00:00:00  (105h lag)
  aggregated query: 31 days in 0.49s
  sample: [{'date': Timestamp('2024-12-01 00:00:00'), 'crashes': '181', 'slippery': '1'}, {'date': Timestamp('2024-12-02 00:00:00'), 'crashes': '266', 'slippery': '1'}, {'date': Timestamp('2024-12-03 00:00:00'), 'crashes': '258', 'slippery': '0'}]


In [28]:
# DOT Traffic Speeds — real-time link speeds across NYC road network
r = requests.get("https://data.cityofnewyork.us/resource/i4gi-tjb9.json", params={
    "$order": "data_as_of DESC",
    "$limit": "500",
}, timeout=30)
r.raise_for_status()
df_dot = pd.DataFrame(r.json())
df_dot["speed"]      = pd.to_numeric(df_dot["speed"], errors="coerce")
df_dot["data_as_of"] = pd.to_datetime(df_dot["data_as_of"])
latest_dot = df_dot["data_as_of"].max()
lag_h = (NOW - latest_dot).total_seconds() / 3600
print(f"DOT speeds:  {latest_dot}  ({lag_h:.1f}h lag)")
print(f"  Links: {len(df_dot)} | Avg speed: {df_dot['speed'].mean():.1f} mph")
print(f"  By borough: {df_dot.groupby('borough')['speed'].mean().round(1).to_dict()}")

DOT speeds:  2026-04-04 08:13:12  (0.8h lag)
  Links: 500 | Avg speed: 34.3 mph
  By borough: {'Bronx': 38.6, 'Brooklyn': 29.1, 'Manhattan': 14.9, 'Queens': 44.0, 'Staten Island': 38.6}


In [29]:
# FloodNet — IoT flood depth sensors, per-event records
# Endpoint: data.cityofnewyork.us/resource/aq7i-eu5q.json
# Lag: event-based (days to weeks after event closes)
# Feature idea: daily binary flood flag + max_depth_inches per day (lag 2-3 days)
r = requests.get("https://data.cityofnewyork.us/resource/aq7i-eu5q.json", params={
    "$order": "flood_start_time DESC",
    "$limit": "200",
}, timeout=30)
r.raise_for_status()
df_flood = pd.DataFrame(r.json())
df_flood["flood_start_time"] = pd.to_datetime(df_flood["flood_start_time"])
df_flood["max_depth_inches"]  = pd.to_numeric(df_flood["max_depth_inches"], errors="coerce")
latest_flood = df_flood["flood_start_time"].max()
lag_h = (NOW - latest_flood).total_seconds() / 3600

# Aggregate to daily: count of flood events and max depth across all sensors
df_flood["date"] = df_flood["flood_start_time"].dt.normalize()
daily_flood = df_flood.groupby("date").agg(
    ft_flood_events=("flood_start_time", "count"),
    ft_flood_max_depth_in=("max_depth_inches", "max"),
).reset_index()

print(f"FloodNet:    {latest_flood}  ({lag_h:.0f}h lag)")
print(f"  Sensors in sample: {df_flood['sensor_name'].nunique()}")
print(f"  Date range: {daily_flood['date'].min().date()} – {daily_flood['date'].max().date()}")
print(f"  Sample daily aggregation:")
print(daily_flood.tail(5).to_string(index=False))

FloodNet:    2026-01-07 15:14:27  (2082h lag)
  Sensors in sample: 130
  Date range: 2025-10-13 – 2026-01-07
  Sample daily aggregation:
      date  ft_flood_events  ft_flood_max_depth_in
2025-12-19                1                   1.73
2025-12-27                3                   6.18
2026-01-03                1                   4.25
2026-01-06                1                   3.90
2026-01-07                2                   3.27


In [30]:
# CitiBike — monthly trip data (S3)
# URL format changed: recent files are YYYYMM-citibike-tripdata.zip (no .csv in name)
# Lag: ~2 weeks after month end | Files are 200-500MB — fetch only if needed
# Feature idea: daily trip count + e-bike share (lag ~14 days)
#
# FEASIBILITY NOTE: NYC-wide monthly files are 200-500MB each. Backfilling 4 years
# would require ~24GB of downloads. For production use, consider the TLC monthly
# aggregate (v6kb-cqej) which covers FHV/taxi at 1-2 month lag but is lightweight.
# This cell tests the schema and lag only using the JC (Jersey City) subset (~5MB).
import io, zipfile

def _fetch_citibike_jc_month(year: int, month: int) -> pd.DataFrame:
    """Fetch Jersey City CitiBike subset — same schema, ~1% of NYC volume, ~5MB."""
    url = f"https://s3.amazonaws.com/tripdata/JC-{year}{month:02d}-citibike-tripdata.csv.zip"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        name = next(n for n in z.namelist() if n.endswith(".csv"))
        df = pd.read_csv(z.open(name), usecols=["started_at", "rideable_type"])
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df["date"] = df["started_at"].dt.normalize()
    return df

# Start from previous month — current month not yet published
df_bike = None
latest_bike_month = None
for p in pd.period_range(end=NOW - pd.DateOffset(months=1), periods=3, freq="M")[::-1]:
    try:
        df_bike = _fetch_citibike_jc_month(p.year, p.month)
        latest_bike_month = p
        break
    except Exception as e:
        print(f"  {p}: {e}")

if df_bike is not None:
    daily_bike = df_bike.groupby("date").agg(
        ft_citibike_trips=("date", "count"),
        ft_citibike_ebike_share=("rideable_type", lambda x: (x == "electric_bike").mean()),
    )
    latest_bike = pd.Timestamp(latest_bike_month.to_timestamp(how="E"))
    lag_h = (NOW - latest_bike).total_seconds() / 3600
    print(f"CitiBike (JC subset): month {latest_bike_month}  ({lag_h:.0f}h lag to month-end)")
    print(f"  Trips in month: {len(df_bike):,} | Days: {daily_bike.shape[0]}")
    print(f"  Daily avg trips: {daily_bike['ft_citibike_trips'].mean():,.0f}")
    print(f"  E-bike share: {daily_bike['ft_citibike_ebike_share'].mean():.1%}")
    print(f"  Schema confirmed: {list(daily_bike.columns)}")
    print(f"  Sample:")
    print(daily_bike.tail(5).to_string())
else:
    print("No CitiBike data fetched")

CitiBike (JC subset): month 2026-03  (81h lag to month-end)
  Trips in month: 62,367 | Days: 32
  Daily avg trips: 1,949
  E-bike share: 65.1%
  Schema confirmed: ['ft_citibike_trips', 'ft_citibike_ebike_share']
  Sample:
            ft_citibike_trips  ft_citibike_ebike_share
date                                                  
2026-03-27               2464                 0.696834
2026-03-28               1786                 0.683651
2026-03-29               1855                 0.656604
2026-03-30               2488                 0.615756
2026-03-31               3118                 0.625722


In [31]:
# Bike & Pedestrian Counts — DOT automated sensors, 15-min granularity
# Endpoint: data.cityofnewyork.us/resource/ct66-47at.json
# Lag: real-time (~15 min)
# Feature idea: daily total bike + pedestrian counts citywide (lag 0 days; real-time signal)
# Caveat: sensor network incomplete — aggregate across available sensors, not a census
r = requests.get("https://data.cityofnewyork.us/resource/ct66-47at.json", params={
    "$where": f"timestamp >= '{(NOW - pd.Timedelta(days=7)).strftime('%Y-%m-%dT00:00:00')}'",
    "$order": "timestamp DESC",
    "$limit": "50000",
}, timeout=60)
r.raise_for_status()
df_ped = pd.DataFrame(r.json())
df_ped["timestamp"] = pd.to_datetime(df_ped["timestamp"])
df_ped["counts"]    = pd.to_numeric(df_ped["counts"], errors="coerce")
df_ped["date"]      = df_ped["timestamp"].dt.normalize()

latest_ped = df_ped["timestamp"].max()
lag_h = (NOW - latest_ped).total_seconds() / 3600

daily_ped = df_ped.groupby(["date", "travelmode"])["counts"].sum().unstack(fill_value=0)
daily_ped.columns = [f"ft_ped_{c.replace(' ', '_')}" for c in daily_ped.columns]

print(f"Bike/Ped:    {latest_ped}  ({lag_h:.1f}h lag)")
print(f"  Sensors: {df_ped['sensor_id'].nunique()} | Modes: {df_ped['travelmode'].unique().tolist()}")
print(f"  Rows fetched: {len(df_ped):,}")
print(f"  Daily totals (last 5 days):")
print(daily_ped.tail(5).to_string())

Bike/Ped:    2026-04-03 05:00:00  (28.0h lag)
  Sensors: 29 | Modes: ['bike', 'pedestrian']
  Rows fetched: 50,000
  Daily totals (last 5 days):
            ft_ped_bike  ft_ped_pedestrian
date                                      
2026-03-30        53924                850
2026-03-31        66516               1424
2026-04-01        58404                914
2026-04-02        34440                412
2026-04-03          841                  6


In [32]:
# MTA Congestion Relief Zone — vehicle entries into Manhattan CBD
# Endpoint: data.ny.gov/resource/t6yz-b64h.json
# Lag: ~3 weeks | Available since Jan 2025 only (short history)
# Feature idea: daily total entries by vehicle class (lag 21 days)
r = requests.get("https://data.ny.gov/resource/t6yz-b64h.json", params={
    "$order": "toll_hour DESC",
    "$limit": "5000",
}, timeout=30)
r.raise_for_status()
df_cz = pd.DataFrame(r.json())
df_cz["toll_hour"]   = pd.to_datetime(df_cz["toll_hour"])
df_cz["crz_entries"] = pd.to_numeric(df_cz["crz_entries"], errors="coerce")
df_cz["date"]        = df_cz["toll_hour"].dt.normalize()

latest_cz = df_cz["toll_hour"].max()
lag_h = (NOW - latest_cz).total_seconds() / 3600

daily_cz = df_cz.groupby(["date", "vehicle_class"])["crz_entries"].sum().unstack(fill_value=0)
daily_cz.columns = [f"ft_cz_{c.lower().replace(' ', '_').replace('/', '_').replace('-', '_')}" for c in daily_cz.columns]
daily_cz["ft_cz_total"] = daily_cz.sum(axis=1)

print(f"Congestion Zone: {latest_cz}  ({lag_h:.0f}h lag)")
print(f"  Vehicle classes: {df_cz['vehicle_class'].unique().tolist()}")
print(f"  Date range: {daily_cz.index.min().date()} – {daily_cz.index.max().date()}")
print(f"  Note: data only available from Jan 2025 — short history for training")
print(f"  Sample (last 5 days):")
print(daily_cz[["ft_cz_total"]].tail(5).to_string())

Congestion Zone: 2026-03-14 23:00:00  (490h lag)
  Vehicle classes: ['4 - Buses', '5 - Motorcycles', 'TLC Taxi/FHV', '1 - Cars, Pickups and Vans', '2 - Single-Unit Trucks', '3 - Multi-Unit Trucks']
  Date range: 2026-03-14 – 2026-03-14
  Note: data only available from Jan 2025 — short history for training
  Sample (last 5 days):
            ft_cz_total
date                   
2026-03-14       267073


In [33]:
# NYC Evictions — marshal-executed evictions, ~1 day lag
# Endpoint: data.cityofnewyork.us/resource/6z8x-wfk4.json
# Lag: ~1 day
# Feature idea: daily eviction count citywide (lag 2 days to be safe)
# Rationale: may carry weak seasonal signal (heating season → more heat violations → evictions)
r = requests.get("https://data.cityofnewyork.us/resource/6z8x-wfk4.json", params={
    "$select": "date_trunc_ymd(executed_date) as date, count(*) as ft_evictions",
    "$group":  "date_trunc_ymd(executed_date)",
    "$where":  f"executed_date >= '{(NOW - pd.Timedelta(days=90)).strftime('%Y-%m-%dT00:00:00')}'",
    "$order":  "date DESC",
    "$limit":  "100",
}, timeout=30)
r.raise_for_status()
df_evict = pd.DataFrame(r.json())
df_evict["date"]         = pd.to_datetime(df_evict["date"]).dt.normalize()
df_evict["ft_evictions"] = pd.to_numeric(df_evict["ft_evictions"], errors="coerce")
df_evict = df_evict.set_index("date").sort_index()

latest_evict = df_evict.index.max()
lag_h = (NOW - latest_evict).total_seconds() / 3600

print(f"Evictions:   {latest_evict.date()}  ({lag_h:.0f}h lag)")
print(f"  Daily avg (90 days): {df_evict['ft_evictions'].mean():.1f}")
print(f"  Daily max: {df_evict['ft_evictions'].max():.0f}")
print(f"  Sample (last 10 days):")
print(df_evict.tail(10).to_string())

Evictions:   2026-04-03  (33h lag)
  Daily avg (90 days): 88.3
  Daily max: 152
  Sample (last 10 days):
            ft_evictions
date                    
2026-03-23            86
2026-03-24            99
2026-03-25            60
2026-03-26            84
2026-03-27            70
2026-03-30            76
2026-03-31            71
2026-04-01            65
2026-04-02            51
2026-04-03             3


In [34]:
sources = {
    "Open-Meteo (ERA5)":         latest_openmeteo,
    "NYISO Zone J":              latest_nyiso,
    "MTA Ridership":             latest_mta_daily,
    "NYC 311":                   latest_311,
    "Crashes":                   latest_crashes,
    "DOT Traffic Speeds":        latest_dot,
    "FloodNet":                  latest_flood,
    "CitiBike Trips":            latest_bike if df_bike is not None else pd.NaT,
    "Bike & Ped Counts":         latest_ped,
    "Congestion Zone":           latest_cz,
    "Evictions":                 latest_evict.index.max() if hasattr(latest_evict, "index") else latest_evict,
}

summary = pd.DataFrame.from_dict(sources, orient="index", columns=["latest_timestamp"])
summary["lag_hours"] = summary["latest_timestamp"].apply(
    lambda t: round((NOW - t).total_seconds() / 3600, 1) if pd.notna(t) else None
)
summary.index.name = "source"
summary

,latest_timestamp,lag_hours
source,,
Open-Meteo (ERA5),2026-04-04 23:00:00.000000,-14.0
NYISO Zone J,2026-04-04 08:45:00.000000,0.3
MTA Ridership,2026-04-02 00:00:00.000000,57.0
NYC 311,2026-04-03 03:04:27.000000,30.0
Crashes,2026-03-31 00:00:00.000000,105.0
DOT Traffic Speeds,2026-04-04 08:13:12.000000,0.8
FloodNet,2026-01-07 15:14:27.000000,2081.8
CitiBike Trips,2026-03-31 23:59:59.999999,81.0
Bike & Ped Counts,2026-04-03 05:00:00.000000,28.0
